In [2]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import warnings


train_df = pd.read_csv(r'D:\Chorme\fashion-mnist_train.csv')
test_df = pd.read_csv(r'D:\Chorme\fashion-mnist_test.csv')
display(train_df.head())

train_df = train_df.dropna()
test_df = test_df.dropna()

X_train = train_df.drop('label', axis=1).values
y_train = train_df['label'].values
X_test = test_df.drop('label', axis=1).values
y_test = test_df['label'].values

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

ml_model_1 = LogisticRegression()
ml_model_1.fit(X_train_scaled, y_train)
y_pred_1 = ml_model_1.predict(X_test_scaled)

ml_model_2 = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
ml_model_2.fit(X_train_scaled, y_train)
y_pred_2 = ml_model_2.predict(X_test_scaled)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

X_train_t = torch.FloatTensor(X_train_scaled).to(device)
y_train_t = torch.LongTensor(y_train.copy()).to(device)
X_test_t = torch.FloatTensor(X_test_scaled).to(device)
y_test_t = torch.LongTensor(y_test).to(device)

train_dataset = TensorDataset(X_train_t, y_train_t)
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)

class UltimateMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(784, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        return self.layers(x)

dl_model = UltimateMLP().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(dl_model.parameters(), lr=0.001, weight_decay=1e-4)

epochs = 30
for epoch in range(epochs):
    dl_model.train()
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        outputs = dl_model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()

with torch.no_grad():
    dl_model.eval()
    dl_outputs = dl_model(X_test_t)
    _, dl_preds = torch.max(dl_outputs, 1)
    final = classification_report(y_test_t.cpu().numpy(), dl_preds.cpu().numpy())

print(f"Logistic Regression: {classification_report(y_test, y_pred_1)}")
print(f"Random Forest: {classification_report(y_test, y_pred_2)}")
print(f"PyTorch Ultimate MLP: {final}")


,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,2,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,6,0,0,0,0,0,0,0,5,0,...,0,0,0,30,43,0,0,0,0,0
3,0,0,0,0,1,2,0,0,0,0,...,3,0,0,0,0,1,0,0,0,0
4,3,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


C:\Users\Admin\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Logistic Regression:               precision    recall  f1-score   support

           0       0.78      0.82      0.80      1000
           1       0.97      0.98      0.97      1000
           2       0.78      0.76      0.77      1000
           3       0.87      0.87      0.87      1000
           4       0.78      0.81      0.79      1000
           5       0.93      0.91      0.92      1000
           6       0.64      0.59      0.61      1000
           7       0.90      0.92      0.91      1000
           8       0.94      0.94      0.94      1000
           9       0.92      0.94      0.93      1000

    accuracy                           0.85     10000
   macro avg       0.85      0.85      0.85     10000
weighted avg       0.85      0.85      0.85     10000

Random Forest:               precision    recall  f1-score   support

           0       0.82      0.86      0.84      1000
           1       0.99      0.97      0.98      1000
           2       0.80      0.80      0.8